In [1]:
# STEP 1: Install all libraries required for the prototype

!pip install -q \
    sentence-transformers \
    rank-bm25 \
    pypdf \
    python-docx \
    spacy \
    scikit-learn \
    pandas \
    numpy \
    matplotlib \
    streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 85.7 MB/s eta 0:00:00


In [2]:
# Download the English NLP model for text processing

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# Test all required imports

import pandas as pd
import numpy as np
import spacy
import sklearn
import streamlit
import torch

from pypdf import PdfReader
from docx import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

print("All required libraries were installed successfully.")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("STEP 1 COMPLETED")

All required libraries were installed successfully.
PyTorch version: 2.11.0+cpu
GPU available: False
STEP 1 COMPLETED


In [6]:
# STEP 2: Generate 20 synthetic Software Engineer CVs

import os
import random
import pandas as pd

from pathlib import Path
from docx import Document


# ---------------------------------------------------------
# 1. Make the results reproducible
# ---------------------------------------------------------

random.seed(42)


# ---------------------------------------------------------
# 2. Create project folders
# ---------------------------------------------------------

BASE_FOLDER = Path("/content/cv_job_matcher")
CV_FOLDER = BASE_FOLDER / "synthetic_cvs"
OUTPUT_FOLDER = BASE_FOLDER / "outputs"

CV_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Project folders created.")


# ---------------------------------------------------------
# 3. Create the Software Engineer job description
# ---------------------------------------------------------

job_description = """
Job Title: Software Engineer

We are seeking a Software Engineer to design, develop, test and
maintain reliable software applications.

The ideal candidate should have experience with Python, SQL,
object-oriented programming, REST APIs, Git, Docker and cloud
technologies such as AWS.

Knowledge of machine learning, Pandas, NumPy, Scikit-learn,
software testing and Agile development is desirable.

The candidate should demonstrate strong problem-solving,
communication, teamwork and software-development skills.

Main responsibilities include:

- Developing and maintaining Python applications
- Creating and integrating REST APIs
- Working with SQL databases
- Using Git for version control
- Testing and debugging software
- Supporting containerised applications using Docker
- Contributing to Agile software-development projects
- Producing clear technical documentation
"""

required_skills = [
    "Python",
    "SQL",
    "Object-Oriented Programming",
    "REST APIs",
    "Git",
    "Docker",
    "AWS",
    "Machine Learning",
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "Software Testing",
    "Agile"
]

# Save the job description
job_description_file = BASE_FOLDER / "software_engineer_job_description.txt"

with open(job_description_file, "w", encoding="utf-8") as file:
    file.write(job_description.strip())

print("Job description created.")


# ---------------------------------------------------------
# 4. Define additional skills for variation
# ---------------------------------------------------------

additional_skills = [
    "Java",
    "JavaScript",
    "Flask",
    "Django",
    "FastAPI",
    "Linux",
    "PostgreSQL",
    "MySQL",
    "TensorFlow",
    "Kubernetes",
    "CI/CD",
    "HTML",
    "CSS",
    "Power BI"
]


# ---------------------------------------------------------
# 5. Define suitability groups
# ---------------------------------------------------------

group_settings = {
    "High": {
        "number": 7,
        "minimum_skills": 9,
        "maximum_skills": 12,
        "minimum_years": 4,
        "maximum_years": 7,
        "role": "Software Engineer",
        "education": "BSc Computer Science"
    },

    "Medium": {
        "number": 7,
        "minimum_skills": 5,
        "maximum_skills": 8,
        "minimum_years": 2,
        "maximum_years": 4,
        "role": "Junior Software Developer",
        "education": "BSc Information Technology"
    },

    "Low": {
        "number": 6,
        "minimum_skills": 1,
        "maximum_skills": 4,
        "minimum_years": 0,
        "maximum_years": 2,
        "role": "IT Support Assistant",
        "education": "Bachelor of Business Administration"
    }
}


# ---------------------------------------------------------
# 6. Functions for creating CV content
# ---------------------------------------------------------

def create_summary(label, years, skills):
    """Create a professional summary based on suitability."""

    selected_skills = ", ".join(skills[:5])

    if label == "High":
        return (
            f"Experienced software engineer with {years} years of "
            f"professional experience developing scalable applications. "
            f"Strong knowledge of {selected_skills}. Experienced in "
            f"backend development, cloud deployment, testing and Agile "
            f"software delivery."
        )

    if label == "Medium":
        return (
            f"Junior software developer with approximately {years} years "
            f"of experience. Familiar with {selected_skills}. Has worked "
            f"on academic and small commercial software projects and is "
            f"seeking to develop stronger software-engineering expertise."
        )

    return (
        f"Entry-level applicant with {years} years of general technology "
        f"experience. Has limited exposure to {selected_skills}. Most "
        f"previous experience has focused on administrative support, "
        f"customer service and basic computer operations."
    )


def create_experience(label, role, years, skills):
    """Create synthetic work-experience statements."""

    if label == "High":
        return [
            f"Developed production software using {skills[0]} and {skills[1]}.",
            "Designed backend services and integrated databases.",
            "Participated in code reviews, testing and debugging.",
            "Worked in an Agile team using version-control tools.",
            "Supported deployment and technical documentation."
        ]

    if label == "Medium":
        return [
            f"Assisted with application development using {skills[0]}.",
            "Completed software testing and debugging tasks.",
            "Worked with senior developers on small software features.",
            "Used version control for academic and junior projects."
        ]

    return [
        "Provided general computer and administrative support.",
        "Installed software and helped users resolve basic technical issues.",
        "Maintained records and responded to customer queries.",
        "Completed introductory online programming courses."
    ]


def create_projects(label, skills):
    """Create synthetic project descriptions."""

    if label == "High":
        return [
            (
                "Candidate Matching Application: Developed a software "
                "application for matching documents using semantic text "
                "analysis and structured data."
            ),
            (
                "Cloud-Based API: Designed and deployed a backend API "
                "connected to a relational database."
            )
        ]

    if label == "Medium":
        return [
            (
                "Student Management System: Created a small application "
                "for storing and retrieving student information."
            ),
            (
                "Data Analysis Project: Cleaned and analysed a structured "
                "dataset using programming libraries."
            )
        ]

    return [
        (
            "Personal Website: Created a basic informational website as "
            "part of an introductory technology course."
        )
    ]


def save_cv_as_docx(
    candidate_id,
    label,
    role,
    years,
    education,
    skills,
    file_path
):
    """Save one synthetic CV as a DOCX file."""

    document = Document()

    document.add_heading(candidate_id, level=0)

    document.add_heading("Professional Profile", level=1)
    document.add_paragraph(
        create_summary(label, years, skills)
    )

    document.add_heading("Technical Skills", level=1)

    for skill in skills:
        document.add_paragraph(skill, style="List Bullet")

    document.add_heading("Professional Experience", level=1)
    document.add_paragraph(
        f"{role} — Synthetic Technology Organisation"
    )
    document.add_paragraph(
        f"Approximate experience: {years} years"
    )

    experience_points = create_experience(
        label,
        role,
        years,
        skills
    )

    for point in experience_points:
        document.add_paragraph(point, style="List Bullet")

    document.add_heading("Projects", level=1)

    for project in create_projects(label, skills):
        document.add_paragraph(project, style="List Bullet")

    document.add_heading("Education", level=1)
    document.add_paragraph(education)

    document.add_heading("Additional Information", level=1)
    document.add_paragraph(
        "This is an anonymised synthetic CV created solely for "
        "academic prototype evaluation."
    )

    document.save(file_path)


# ---------------------------------------------------------
# 7. Generate all 20 CVs
# ---------------------------------------------------------

ground_truth_records = []
candidate_number = 1

for label, settings in group_settings.items():

    for _ in range(settings["number"]):

        candidate_id = f"Candidate_{candidate_number:02d}"

        number_of_required_skills = random.randint(
            settings["minimum_skills"],
            settings["maximum_skills"]
        )

        number_of_required_skills = min(
            number_of_required_skills,
            len(required_skills)
        )

        selected_required_skills = random.sample(
            required_skills,
            number_of_required_skills
        )

        number_of_extra_skills = random.randint(1, 3)

        selected_extra_skills = random.sample(
            additional_skills,
            number_of_extra_skills
        )

        candidate_skills = (
            selected_required_skills
            + selected_extra_skills
        )

        years_experience = random.randint(
            settings["minimum_years"],
            settings["maximum_years"]
        )

        filename = f"{candidate_id}.docx"
        file_path = CV_FOLDER / filename

        save_cv_as_docx(
            candidate_id=candidate_id,
            label=label,
            role=settings["role"],
            years=years_experience,
            education=settings["education"],
            skills=candidate_skills,
            file_path=file_path
        )

        ground_truth_records.append({
            "Candidate ID": candidate_id,
            "File Name": filename,
            "Ground Truth Label": label,
            "Years of Experience": years_experience,
            "Number of Required Skills": len(
                selected_required_skills
            ),
            "Required Skills Present": ", ".join(
                selected_required_skills
            )
        })

        candidate_number += 1


# ---------------------------------------------------------
# 8. Save the ground-truth labels
# ---------------------------------------------------------

ground_truth_df = pd.DataFrame(ground_truth_records)

ground_truth_file = BASE_FOLDER / "ground_truth_labels.csv"

ground_truth_df.to_csv(
    ground_truth_file,
    index=False
)


# ---------------------------------------------------------
# 9. Show completion information
# ---------------------------------------------------------

created_files = sorted(CV_FOLDER.glob("*.docx"))

print("\nSynthetic CV generation completed.")
print("Total CVs created:", len(created_files))
print("CV folder:", CV_FOLDER)
print("Ground-truth file:", ground_truth_file)
print("Job-description file:", job_description_file)

print("\nSuitability distribution:")
print(
    ground_truth_df["Ground Truth Label"].value_counts()
)

print("\nSTEP 2 COMPLETED")

Project folders created.
Job description created.

Synthetic CV generation completed.
Total CVs created: 20
CV folder: /content/cv_job_matcher/synthetic_cvs
Ground-truth file: /content/cv_job_matcher/ground_truth_labels.csv
Job-description file: /content/cv_job_matcher/software_engineer_job_description.txt

Suitability distribution:
Ground Truth Label
High      7
Medium    7
Low       6
Name: count, dtype: int64

STEP 2 COMPLETED


In [7]:
# STEP 3: Extract and preprocess text from all synthetic CVs

import re
import pandas as pd

from pathlib import Path
from docx import Document
from pypdf import PdfReader


# ---------------------------------------------------------
# 1. Extract text from a DOCX file
# ---------------------------------------------------------

def extract_text_from_docx(file_path):
    """
    Extract readable text from a Microsoft Word DOCX file.
    """

    document = Document(file_path)

    paragraphs = [
        paragraph.text.strip()
        for paragraph in document.paragraphs
        if paragraph.text.strip()
    ]

    return "\n".join(paragraphs)


# ---------------------------------------------------------
# 2. Extract text from a PDF file
# ---------------------------------------------------------

def extract_text_from_pdf(file_path):
    """
    Extract readable text from a text-based PDF file.
    """

    reader = PdfReader(str(file_path))
    pages_text = []

    for page in reader.pages:
        page_text = page.extract_text() or ""

        if page_text.strip():
            pages_text.append(page_text.strip())

    return "\n".join(pages_text)


# ---------------------------------------------------------
# 3. Automatically identify the CV file type
# ---------------------------------------------------------

def extract_cv_text(file_path):
    """
    Extract text from either a DOCX or PDF CV.
    """

    file_path = Path(file_path)
    extension = file_path.suffix.lower()

    if extension == ".docx":
        return extract_text_from_docx(file_path)

    elif extension == ".pdf":
        return extract_text_from_pdf(file_path)

    else:
        raise ValueError(
            f"Unsupported CV format: {extension}"
        )


# ---------------------------------------------------------
# 4. Clean and preprocess the extracted text
# ---------------------------------------------------------

def preprocess_text(text):
    """
    Clean CV text before semantic and skill analysis.
    """

    if not isinstance(text, str):
        return ""

    # Convert text to lowercase
    text = text.lower()

    # Replace line breaks and tabs with spaces
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # Remove email addresses
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Remove website links
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove phone-number patterns
    text = re.sub(
        r"\+?\d[\d\s().-]{7,}\d",
        " ",
        text
    )

    # Keep useful technical symbols such as +, #, . and -
    text = re.sub(
        r"[^a-z0-9+#.\-/\s]",
        " ",
        text
    )

    # Remove repeated spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ---------------------------------------------------------
# 5. Read all CV files from the synthetic CV folder
# ---------------------------------------------------------

supported_files = sorted([
    file_path
    for file_path in CV_FOLDER.iterdir()
    if file_path.suffix.lower() in [".docx", ".pdf"]
])

cv_records = []

for file_path in supported_files:

    try:
        raw_text = extract_cv_text(file_path)
        clean_text = preprocess_text(raw_text)

        cv_records.append({
            "Candidate ID": file_path.stem,
            "File Name": file_path.name,
            "Raw Text": raw_text,
            "Clean Text": clean_text,
            "Word Count": len(clean_text.split()),
            "Extraction Status": "Successful"
        })

        print(
            f"Processed: {file_path.name} "
            f"({len(clean_text.split())} words)"
        )

    except Exception as error:

        cv_records.append({
            "Candidate ID": file_path.stem,
            "File Name": file_path.name,
            "Raw Text": "",
            "Clean Text": "",
            "Word Count": 0,
            "Extraction Status": f"Failed: {error}"
        })

        print(
            f"Failed to process {file_path.name}: {error}"
        )


# ---------------------------------------------------------
# 6. Convert the extracted results into a dataframe
# ---------------------------------------------------------

cv_text_df = pd.DataFrame(cv_records)


# ---------------------------------------------------------
# 7. Add the ground-truth labels
# ---------------------------------------------------------

cv_data_df = cv_text_df.merge(
    ground_truth_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Years of Experience",
            "Number of Required Skills"
        ]
    ],
    on="Candidate ID",
    how="left"
)


# ---------------------------------------------------------
# 8. Save the extracted dataset
# ---------------------------------------------------------

extracted_data_file = (
    OUTPUT_FOLDER / "extracted_cv_dataset.csv"
)

cv_data_df.to_csv(
    extracted_data_file,
    index=False
)


# ---------------------------------------------------------
# 9. Check the results
# ---------------------------------------------------------

successful_count = (
    cv_data_df["Extraction Status"]
    == "Successful"
).sum()

failed_count = len(cv_data_df) - successful_count

print("\nCV text extraction completed.")
print("Total CV files:", len(cv_data_df))
print("Successfully processed:", successful_count)
print("Failed:", failed_count)
print("Saved dataset:", extracted_data_file)


# Display a summary table

display(
    cv_data_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Years of Experience",
            "Number of Required Skills",
            "Word Count",
            "Extraction Status"
        ]
    ]
)


# Final validation

if (
    len(cv_data_df) == 20
    and successful_count == 20
):
    print("\nAll 20 CVs were extracted successfully.")
    print("STEP 3 COMPLETED")
else:
    print(
        "\nWarning: Some CVs were not processed correctly."
    )

Processed: Candidate_01.docx (146 words)
Processed: Candidate_02.docx (144 words)
Processed: Candidate_03.docx (145 words)
Processed: Candidate_04.docx (150 words)
Processed: Candidate_05.docx (145 words)
Processed: Candidate_06.docx (146 words)
Processed: Candidate_07.docx (150 words)
Processed: Candidate_08.docx (137 words)
Processed: Candidate_09.docx (138 words)
Processed: Candidate_10.docx (132 words)
Processed: Candidate_11.docx (133 words)
Processed: Candidate_12.docx (137 words)
Processed: Candidate_13.docx (135 words)
Processed: Candidate_14.docx (135 words)
Processed: Candidate_15.docx (118 words)
Processed: Candidate_16.docx (120 words)
Processed: Candidate_17.docx (120 words)
Processed: Candidate_18.docx (116 words)
Processed: Candidate_19.docx (116 words)
Processed: Candidate_20.docx (118 words)

CV text extraction completed.
Total CV files: 20
Successfully processed: 20
Failed: 0
Saved dataset: /content/cv_job_matcher/outputs/extracted_cv_dataset.csv


,Candidate ID,Ground Truth Label,Years of Experience,Number of Required Skills,Word Count,Extraction Status
0,Candidate_01,High,7,9,146,Successful
1,Candidate_02,High,6,9,144,Successful
2,Candidate_03,High,4,9,145,Successful
3,Candidate_04,High,5,11,150,Successful
4,Candidate_05,High,6,9,145,Successful
5,Candidate_06,High,5,10,146,Successful
6,Candidate_07,High,7,11,150,Successful
7,Candidate_08,Medium,3,6,137,Successful
8,Candidate_09,Medium,4,7,138,Successful
9,Candidate_10,Medium,4,6,132,Successful



All 20 CVs were extracted successfully.
STEP 3 COMPLETED


In [10]:
# STEP 4: Sentence-BERT semantic similarity using a smaller model

import os
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer


# ---------------------------------------------------------
# 1. Prevent the Hugging Face download from getting stuck
# ---------------------------------------------------------

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


# ---------------------------------------------------------
# 2. Load a smaller Sentence-BERT model
# ---------------------------------------------------------

MODEL_NAME = "paraphrase-MiniLM-L3-v2"

print("Downloading and loading Sentence-BERT model...")

sbert_model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

print("Model loaded successfully:", MODEL_NAME)


# ---------------------------------------------------------
# 3. Prepare the job description
# ---------------------------------------------------------

clean_job_description = preprocess_text(job_description)

if not clean_job_description:
    raise ValueError(
        "The job description is empty."
    )

print(
    "Job description word count:",
    len(clean_job_description.split())
)


# ---------------------------------------------------------
# 4. Prepare all 20 CV texts
# ---------------------------------------------------------

candidate_texts = (
    cv_data_df["Clean Text"]
    .fillna("")
    .astype(str)
    .tolist()
)

empty_cv_count = sum(
    1
    for text in candidate_texts
    if not text.strip()
)

if empty_cv_count > 0:
    raise ValueError(
        f"{empty_cv_count} CVs contain no readable text."
    )

print("CVs ready for analysis:", len(candidate_texts))


# ---------------------------------------------------------
# 5. Generate embeddings
# ---------------------------------------------------------

print("Generating semantic embeddings...")

job_embedding = sbert_model.encode(
    clean_job_description,
    convert_to_numpy=True,
    normalize_embeddings=True
)

cv_embeddings = sbert_model.encode(
    candidate_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Job embedding shape:", job_embedding.shape)
print("CV embeddings shape:", cv_embeddings.shape)


# ---------------------------------------------------------
# 6. Calculate cosine similarity
# ---------------------------------------------------------

semantic_similarities = np.dot(
    cv_embeddings,
    job_embedding
)

cv_data_df["Semantic Similarity"] = (
    semantic_similarities
)

cv_data_df["Semantic Score (%)"] = (
    np.clip(
        semantic_similarities,
        0,
        1
    ) * 100
)


# ---------------------------------------------------------
# 7. Create the semantic ranking
# ---------------------------------------------------------

semantic_results_df = (
    cv_data_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Years of Experience",
            "Number of Required Skills",
            "Semantic Similarity",
            "Semantic Score (%)"
        ]
    ]
    .sort_values(
        by="Semantic Similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

semantic_results_df.insert(
    0,
    "Semantic Rank",
    range(1, len(semantic_results_df) + 1)
)


# ---------------------------------------------------------
# 8. Round values
# ---------------------------------------------------------

semantic_results_df["Semantic Similarity"] = (
    semantic_results_df[
        "Semantic Similarity"
    ].round(4)
)

semantic_results_df["Semantic Score (%)"] = (
    semantic_results_df[
        "Semantic Score (%)"
    ].round(2)
)


# ---------------------------------------------------------
# 9. Save the results
# ---------------------------------------------------------

semantic_output_file = (
    OUTPUT_FOLDER /
    "sentence_bert_semantic_results.csv"
)

semantic_results_df.to_csv(
    semantic_output_file,
    index=False
)


# ---------------------------------------------------------
# 10. Display the results
# ---------------------------------------------------------

print("\nSentence-BERT analysis completed.")
print("Total candidates ranked:", len(semantic_results_df))
print("Results saved to:", semantic_output_file)

display(semantic_results_df)


# ---------------------------------------------------------
# 11. Final validation
# ---------------------------------------------------------

if (
    len(semantic_results_df) == 20
    and semantic_results_df[
        "Semantic Similarity"
    ].notna().all()
):
    print("\nAll 20 CVs received semantic scores.")
    print("STEP 4 COMPLETED")

else:
    print(
        "\nWarning: Some candidates did not receive scores."
    )

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

Model loaded successfully: paraphrase-MiniLM-L3-v2
Job description word count: 113
CVs ready for analysis: 20
Generating semantic embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Job embedding shape: (384,)
CV embeddings shape: (20, 384)

Sentence-BERT analysis completed.
Total candidates ranked: 20
Results saved to: /content/cv_job_matcher/outputs/sentence_bert_semantic_results.csv


,Semantic Rank,Candidate ID,Ground Truth Label,Years of Experience,Number of Required Skills,Semantic Similarity,Semantic Score (%)
0,1,Candidate_02,High,6,9,0.7685,76.849998
1,2,Candidate_01,High,7,9,0.7670,76.699997
2,3,Candidate_03,High,4,9,0.7616,76.160004
3,4,Candidate_08,Medium,3,6,0.7612,76.120003
4,5,Candidate_14,Medium,2,7,0.7593,75.930000
5,6,Candidate_04,High,5,11,0.7526,75.260002
6,7,Candidate_06,High,5,10,0.7442,74.419998
7,8,Candidate_05,High,6,9,0.7432,74.320000
8,9,Candidate_07,High,7,11,0.7388,73.879997
9,10,Candidate_13,Medium,2,6,0.7343,73.430000



All 20 CVs received semantic scores.
STEP 4 COMPLETED


In [11]:
# STEP 5: Structured skill matching for all 20 CVs

import re
import pandas as pd


# ---------------------------------------------------------
# 1. Define alternative expressions for required skills
# ---------------------------------------------------------

SKILL_ALIASES = {
    "Python": [
        "python"
    ],

    "SQL": [
        "sql",
        "mysql",
        "postgresql",
        "structured query language"
    ],

    "Object-Oriented Programming": [
        "object-oriented programming",
        "object oriented programming",
        "oop"
    ],

    "REST APIs": [
        "rest api",
        "rest apis",
        "restful api",
        "restful apis",
        "api development"
    ],

    "Git": [
        "git",
        "github",
        "gitlab",
        "version control"
    ],

    "Docker": [
        "docker",
        "containerisation",
        "containerization",
        "containerised",
        "containerized"
    ],

    "AWS": [
        "aws",
        "amazon web services"
    ],

    "Machine Learning": [
        "machine learning",
        "predictive modelling",
        "predictive modeling"
    ],

    "Pandas": [
        "pandas"
    ],

    "NumPy": [
        "numpy"
    ],

    "Scikit-learn": [
        "scikit-learn",
        "scikit learn",
        "sklearn"
    ],

    "Software Testing": [
        "software testing",
        "unit testing",
        "integration testing",
        "test automation",
        "testing and debugging"
    ],

    "Agile": [
        "agile",
        "scrum",
        "sprint planning"
    ]
}


# ---------------------------------------------------------
# 2. Normalise text for skill matching
# ---------------------------------------------------------

def normalise_skill_text(text):
    """
    Prepare CV text for structured skill matching.
    """

    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = text.replace("–", "-")
    text = text.replace("—", "-")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ---------------------------------------------------------
# 3. Check whether one skill appears in the CV
# ---------------------------------------------------------

def skill_is_present(cv_text, skill):
    """
    Check the canonical skill and its alternative expressions.
    """

    clean_text = normalise_skill_text(cv_text)

    aliases = SKILL_ALIASES.get(
        skill,
        [skill.lower()]
    )

    for alias in aliases:

        alias = alias.lower().strip()

        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias)
            + r"(?![a-z0-9])"
        )

        if re.search(pattern, clean_text):
            return True

    return False


# ---------------------------------------------------------
# 4. Calculate skill matching for one CV
# ---------------------------------------------------------

def calculate_skill_match(cv_text, required_skills):
    """
    Return matched skills, missing skills and match score.
    """

    matched_skills = []
    missing_skills = []

    for skill in required_skills:

        if skill_is_present(cv_text, skill):
            matched_skills.append(skill)

        else:
            missing_skills.append(skill)

    total_required = len(required_skills)

    if total_required > 0:
        skill_score = (
            len(matched_skills) / total_required
        ) * 100

    else:
        skill_score = 0.0

    return {
        "Matched Skills": matched_skills,
        "Missing Skills": missing_skills,
        "Matched Skill Count": len(matched_skills),
        "Required Skill Count": total_required,
        "Skill Score (%)": skill_score
    }


# ---------------------------------------------------------
# 5. Analyse all 20 CVs
# ---------------------------------------------------------

skill_records = []

for _, row in cv_data_df.iterrows():

    candidate_id = row["Candidate ID"]
    cv_text = row["Clean Text"]

    result = calculate_skill_match(
        cv_text=cv_text,
        required_skills=required_skills
    )

    skill_records.append({
        "Candidate ID": candidate_id,

        "Ground Truth Label":
            row["Ground Truth Label"],

        "Years of Experience":
            row["Years of Experience"],

        "Matched Skill Count":
            result["Matched Skill Count"],

        "Required Skill Count":
            result["Required Skill Count"],

        "Skill Score (%)":
            result["Skill Score (%)"],

        "Matched Skills":
            ", ".join(result["Matched Skills"]),

        "Missing Skills":
            ", ".join(result["Missing Skills"])
    })

    print(
        f"{candidate_id}: "
        f"{result['Matched Skill Count']}/"
        f"{result['Required Skill Count']} skills matched"
    )


# ---------------------------------------------------------
# 6. Convert results into a dataframe
# ---------------------------------------------------------

skill_results_df = pd.DataFrame(
    skill_records
)


# ---------------------------------------------------------
# 7. Rank candidates by skill score
# ---------------------------------------------------------

skill_results_df = (
    skill_results_df
    .sort_values(
        by=[
            "Skill Score (%)",
            "Years of Experience"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

skill_results_df.insert(
    0,
    "Skill Rank",
    range(1, len(skill_results_df) + 1)
)

skill_results_df["Skill Score (%)"] = (
    skill_results_df[
        "Skill Score (%)"
    ].round(2)
)


# ---------------------------------------------------------
# 8. Add skill results to the main CV dataframe
# ---------------------------------------------------------

cv_data_df = cv_data_df.drop(
    columns=[
        "Matched Skill Count",
        "Required Skill Count",
        "Skill Score (%)",
        "Matched Skills",
        "Missing Skills"
    ],
    errors="ignore"
)

cv_data_df = cv_data_df.merge(
    skill_results_df[
        [
            "Candidate ID",
            "Matched Skill Count",
            "Required Skill Count",
            "Skill Score (%)",
            "Matched Skills",
            "Missing Skills"
        ]
    ],
    on="Candidate ID",
    how="left"
)


# ---------------------------------------------------------
# 9. Save the results
# ---------------------------------------------------------

skill_output_file = (
    OUTPUT_FOLDER /
    "structured_skill_matching_results.csv"
)

skill_results_df.to_csv(
    skill_output_file,
    index=False
)


# ---------------------------------------------------------
# 10. Display the skill ranking
# ---------------------------------------------------------

print("\nStructured skill matching completed.")
print("Total candidates analysed:", len(skill_results_df))
print("Results saved to:", skill_output_file)

display(
    skill_results_df[
        [
            "Skill Rank",
            "Candidate ID",
            "Ground Truth Label",
            "Matched Skill Count",
            "Required Skill Count",
            "Skill Score (%)",
            "Matched Skills",
            "Missing Skills"
        ]
    ]
)


# ---------------------------------------------------------
# 11. Validate Step 5
# ---------------------------------------------------------

if (
    len(skill_results_df) == 20
    and skill_results_df[
        "Skill Score (%)"
    ].notna().all()
):
    print("\nAll 20 CVs received skill-matching scores.")
    print("STEP 5 COMPLETED")

else:
    print(
        "\nWarning: Skill matching was not completed "
        "for every candidate."
    )

Candidate_01: 9/13 skills matched
Candidate_02: 9/13 skills matched
Candidate_03: 9/13 skills matched
Candidate_04: 12/13 skills matched
Candidate_05: 9/13 skills matched
Candidate_06: 10/13 skills matched
Candidate_07: 11/13 skills matched
Candidate_08: 8/13 skills matched
Candidate_09: 9/13 skills matched
Candidate_10: 7/13 skills matched
Candidate_11: 8/13 skills matched
Candidate_12: 7/13 skills matched
Candidate_13: 9/13 skills matched
Candidate_14: 8/13 skills matched
Candidate_15: 2/13 skills matched
Candidate_16: 4/13 skills matched
Candidate_17: 4/13 skills matched
Candidate_18: 1/13 skills matched
Candidate_19: 1/13 skills matched
Candidate_20: 3/13 skills matched

Structured skill matching completed.
Total candidates analysed: 20
Results saved to: /content/cv_job_matcher/outputs/structured_skill_matching_results.csv


,Skill Rank,Candidate ID,Ground Truth Label,Matched Skill Count,Required Skill Count,Skill Score (%),Matched Skills,Missing Skills
0,1,Candidate_04,High,12,13,92.31,"Python, SQL, Object-Oriented Programming, REST...",Pandas
1,2,Candidate_07,High,11,13,84.62,"Python, SQL, Object-Oriented Programming, REST...","Git, Scikit-learn"
2,3,Candidate_06,High,10,13,76.92,"SQL, Object-Oriented Programming, REST APIs, G...","Python, Machine Learning, Scikit-learn"
3,4,Candidate_01,High,9,13,69.23,"Python, Object-Oriented Programming, REST APIs...","SQL, AWS, Machine Learning, Pandas"
4,5,Candidate_02,High,9,13,69.23,"Python, SQL, REST APIs, Git, AWS, Pandas, Scik...","Object-Oriented Programming, Docker, Machine L..."
5,6,Candidate_05,High,9,13,69.23,"Python, SQL, REST APIs, Git, AWS, Machine Lear...","Object-Oriented Programming, Docker, Pandas, N..."
6,7,Candidate_03,High,9,13,69.23,"Python, SQL, Object-Oriented Programming, Git,...","REST APIs, Pandas, NumPy, Scikit-learn"
7,8,Candidate_09,Medium,9,13,69.23,"Python, SQL, Object-Oriented Programming, REST...","Docker, NumPy, Scikit-learn, Agile"
8,9,Candidate_13,Medium,9,13,69.23,"Python, SQL, Object-Oriented Programming, Git,...","REST APIs, AWS, Machine Learning, Scikit-learn"
9,10,Candidate_08,Medium,8,13,61.54,"SQL, Object-Oriented Programming, REST APIs, G...","Python, Docker, AWS, Machine Learning, Scikit-..."



All 20 CVs received skill-matching scores.
STEP 5 COMPLETED


In [12]:
# STEP 6: Weighted scoring, candidate ranking and explanations

import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 1. Confirm that the required columns are available
# ---------------------------------------------------------

required_columns = [
    "Candidate ID",
    "Ground Truth Label",
    "Semantic Score (%)",
    "Skill Score (%)",
    "Matched Skills",
    "Missing Skills"
]

missing_columns = [
    column
    for column in required_columns
    if column not in cv_data_df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing: "
        + ", ".join(missing_columns)
        + ". Please rerun Steps 4 and 5."
    )

print("All required scoring columns are available.")


# ---------------------------------------------------------
# 2. Define the scoring weights
# ---------------------------------------------------------

SEMANTIC_WEIGHT = 0.70
SKILL_WEIGHT = 0.30

if not np.isclose(
    SEMANTIC_WEIGHT + SKILL_WEIGHT,
    1.0
):
    raise ValueError(
        "The scoring weights must add up to 1.0."
    )

print("Semantic similarity weight:", SEMANTIC_WEIGHT)
print("Skill matching weight:", SKILL_WEIGHT)


# ---------------------------------------------------------
# 3. Calculate the final weighted score
# ---------------------------------------------------------

cv_data_df["Final Score (%)"] = (
    SEMANTIC_WEIGHT
    * cv_data_df["Semantic Score (%)"]
    +
    SKILL_WEIGHT
    * cv_data_df["Skill Score (%)"]
)

cv_data_df["Final Score (%)"] = (
    cv_data_df["Final Score (%)"]
    .clip(lower=0, upper=100)
    .round(2)
)


# ---------------------------------------------------------
# 4. Classify candidate suitability
# ---------------------------------------------------------

def classify_suitability(final_score):
    """
    Convert the final score into a suitability category.

    These are initial prototype thresholds.
    They will later be evaluated against the ground-truth labels.
    """

    if final_score >= 70:
        return "High"

    elif final_score >= 50:
        return "Medium"

    else:
        return "Low"


cv_data_df["Predicted Suitability"] = (
    cv_data_df["Final Score (%)"]
    .apply(classify_suitability)
)


# ---------------------------------------------------------
# 5. Create a short recommendation
# ---------------------------------------------------------

def create_recommendation(predicted_label):
    """
    Generate a human-centred recommendation.
    """

    if predicted_label == "High":
        return (
            "Strong candidate for initial recruiter review."
        )

    elif predicted_label == "Medium":
        return (
            "Potential candidate, but missing requirements "
            "should be reviewed by a recruiter."
        )

    else:
        return (
            "Limited match for this role based on the current "
            "job description."
        )


cv_data_df["Recommendation"] = (
    cv_data_df["Predicted Suitability"]
    .apply(create_recommendation)
)


# ---------------------------------------------------------
# 6. Generate an explanation for every candidate
# ---------------------------------------------------------

def generate_explanation(row):
    """
    Explain the semantic score, skill score, matched skills,
    missing skills and final recommendation.
    """

    matched_skills = row["Matched Skills"]

    if pd.isna(matched_skills) or not str(matched_skills).strip():
        matched_skills = "No required skills identified"

    missing_skills = row["Missing Skills"]

    if pd.isna(missing_skills) or not str(missing_skills).strip():
        missing_skills = "No required skills missing"

    explanation = (
        f"The candidate received a semantic similarity score of "
        f"{row['Semantic Score (%)']:.2f}% and a structured skill "
        f"match score of {row['Skill Score (%)']:.2f}%. "
        f"The final score was calculated using 70% semantic "
        f"similarity and 30% skill matching. "
        f"Matched skills: {matched_skills}. "
        f"Missing skills: {missing_skills}. "
        f"Predicted suitability: "
        f"{row['Predicted Suitability']}. "
        f"{row['Recommendation']}"
    )

    return explanation


cv_data_df["Explanation"] = (
    cv_data_df.apply(
        generate_explanation,
        axis=1
    )
)


# ---------------------------------------------------------
# 7. Create the final candidate ranking
# ---------------------------------------------------------

final_results_df = (
    cv_data_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Years of Experience",
            "Semantic Score (%)",
            "Skill Score (%)",
            "Final Score (%)",
            "Predicted Suitability",
            "Matched Skills",
            "Missing Skills",
            "Recommendation",
            "Explanation"
        ]
    ]
    .sort_values(
        by=[
            "Final Score (%)",
            "Skill Score (%)",
            "Years of Experience"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

final_results_df.insert(
    0,
    "Final Rank",
    range(1, len(final_results_df) + 1)
)


# ---------------------------------------------------------
# 8. Compare predicted and ground-truth categories
# ---------------------------------------------------------

final_results_df["Prediction Correct"] = (
    final_results_df["Ground Truth Label"]
    ==
    final_results_df["Predicted Suitability"]
)

correct_predictions = (
    final_results_df["Prediction Correct"]
    .sum()
)

total_predictions = len(final_results_df)

initial_accuracy = (
    correct_predictions / total_predictions
) * 100


# ---------------------------------------------------------
# 9. Save the final results
# ---------------------------------------------------------

final_output_file = (
    OUTPUT_FOLDER /
    "final_explainable_candidate_ranking.csv"
)

final_results_df.to_csv(
    final_output_file,
    index=False
)


# ---------------------------------------------------------
# 10. Display the final ranking
# ---------------------------------------------------------

print("\nFinal explainable ranking completed.")
print("Total candidates ranked:", total_predictions)
print(
    "Initial classification accuracy:",
    f"{initial_accuracy:.2f}%"
)
print("Results saved to:", final_output_file)

display(
    final_results_df[
        [
            "Final Rank",
            "Candidate ID",
            "Ground Truth Label",
            "Semantic Score (%)",
            "Skill Score (%)",
            "Final Score (%)",
            "Predicted Suitability",
            "Prediction Correct"
        ]
    ]
)


# ---------------------------------------------------------
# 11. Display the explanation for the top candidate
# ---------------------------------------------------------

top_candidate = final_results_df.iloc[0]

print("\nTOP-RANKED CANDIDATE")
print("--------------------")
print("Candidate:", top_candidate["Candidate ID"])
print("Final rank:", top_candidate["Final Rank"])
print(
    "Final score:",
    f"{top_candidate['Final Score (%)']:.2f}%"
)
print("\nExplanation:")
print(top_candidate["Explanation"])


# ---------------------------------------------------------
# 12. Validate Step 6
# ---------------------------------------------------------

if (
    len(final_results_df) == 20
    and final_results_df[
        "Final Score (%)"
    ].notna().all()
    and final_results_df[
        "Explanation"
    ].notna().all()
):
    print("\nAll candidates received final scores and explanations.")
    print("STEP 6 COMPLETED")

else:
    print(
        "\nWarning: Final scoring was not completed "
        "for every candidate."
    )

All required scoring columns are available.
Semantic similarity weight: 0.7
Skill matching weight: 0.3

Final explainable ranking completed.
Total candidates ranked: 20
Initial classification accuracy: 80.00%
Results saved to: /content/cv_job_matcher/outputs/final_explainable_candidate_ranking.csv


,Final Rank,Candidate ID,Ground Truth Label,Semantic Score (%),Skill Score (%),Final Score (%),Predicted Suitability,Prediction Correct
0,1,Candidate_04,High,75.264168,92.31,80.38,High,True
1,2,Candidate_07,High,73.881340,84.62,77.10,High,True
2,3,Candidate_06,High,74.421249,76.92,75.17,High,True
3,4,Candidate_02,High,76.849213,69.23,74.56,High,True
4,5,Candidate_01,High,76.701828,69.23,74.46,High,True
5,6,Candidate_03,High,76.160095,69.23,74.08,High,True
6,7,Candidate_05,High,74.315437,69.23,72.79,High,True
7,8,Candidate_13,Medium,73.431992,69.23,72.17,High,False
8,9,Candidate_08,Medium,76.119255,61.54,71.75,High,False
9,10,Candidate_14,Medium,75.933334,61.54,71.62,High,False



TOP-RANKED CANDIDATE
--------------------
Candidate: Candidate_04
Final rank: 1
Final score: 80.38%

Explanation:
The candidate received a semantic similarity score of 75.26% and a structured skill match score of 92.31%. The final score was calculated using 70% semantic similarity and 30% skill matching. Matched skills: Python, SQL, Object-Oriented Programming, REST APIs, Git, Docker, AWS, Machine Learning, NumPy, Scikit-learn, Software Testing, Agile. Missing skills: Pandas. Predicted suitability: High. Strong candidate for initial recruiter review.

All candidates received final scores and explanations.
STEP 6 COMPLETED


In [13]:
# STEP 7: BM25 baseline and ranking comparison

import re
import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score


# ---------------------------------------------------------
# 1. Check that previous results are available
# ---------------------------------------------------------

required_columns = [
    "Candidate ID",
    "Clean Text",
    "Ground Truth Label",
    "Final Score (%)"
]

missing_columns = [
    column
    for column in required_columns
    if column not in cv_data_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns: "
        + ", ".join(missing_columns)
        + ". Please rerun the previous steps."
    )

print("All required data are available.")


# ---------------------------------------------------------
# 2. Tokenise text for BM25
# ---------------------------------------------------------

def bm25_tokenize(text):
    """
    Convert text into simple lowercase word tokens.
    """

    if not isinstance(text, str):
        return []

    text = text.lower()

    # Retain common technical characters
    tokens = re.findall(
        r"[a-z0-9+#.\-]+",
        text
    )

    return tokens


# ---------------------------------------------------------
# 3. Prepare the CV corpus
# ---------------------------------------------------------

candidate_ids = (
    cv_data_df["Candidate ID"]
    .astype(str)
    .tolist()
)

candidate_texts = (
    cv_data_df["Clean Text"]
    .fillna("")
    .astype(str)
    .tolist()
)

tokenized_corpus = [
    bm25_tokenize(text)
    for text in candidate_texts
]

tokenized_job_description = bm25_tokenize(
    preprocess_text(job_description)
)

if not tokenized_job_description:
    raise ValueError(
        "The job description contains no usable tokens."
    )

print("CVs prepared for BM25:", len(tokenized_corpus))
print(
    "Job-description tokens:",
    len(tokenized_job_description)
)


# ---------------------------------------------------------
# 4. Build the BM25 model
# ---------------------------------------------------------

bm25_model = BM25Okapi(
    tokenized_corpus
)

bm25_raw_scores = bm25_model.get_scores(
    tokenized_job_description
)

print("BM25 scoring completed.")


# ---------------------------------------------------------
# 5. Normalise BM25 scores for easier display
# ---------------------------------------------------------

minimum_score = float(np.min(bm25_raw_scores))
maximum_score = float(np.max(bm25_raw_scores))

if maximum_score > minimum_score:
    bm25_display_scores = (
        (
            bm25_raw_scores - minimum_score
        )
        /
        (
            maximum_score - minimum_score
        )
    ) * 100
else:
    bm25_display_scores = np.zeros_like(
        bm25_raw_scores
    )


# ---------------------------------------------------------
# 6. Create the BM25 results dataframe
# ---------------------------------------------------------

bm25_results_df = pd.DataFrame({
    "Candidate ID": candidate_ids,
    "Ground Truth Label":
        cv_data_df["Ground Truth Label"].tolist(),
    "BM25 Raw Score": bm25_raw_scores,
    "BM25 Score (%)": bm25_display_scores
})

bm25_results_df = (
    bm25_results_df
    .sort_values(
        by="BM25 Raw Score",
        ascending=False
    )
    .reset_index(drop=True)
)

bm25_results_df.insert(
    0,
    "BM25 Rank",
    range(1, len(bm25_results_df) + 1)
)

bm25_results_df["BM25 Raw Score"] = (
    bm25_results_df["BM25 Raw Score"]
    .round(4)
)

bm25_results_df["BM25 Score (%)"] = (
    bm25_results_df["BM25 Score (%)"]
    .round(2)
)


# ---------------------------------------------------------
# 7. Create a direct ranking comparison
# ---------------------------------------------------------

final_system_ranking = (
    final_results_df[
        [
            "Candidate ID",
            "Final Rank",
            "Final Score (%)",
            "Predicted Suitability"
        ]
    ]
)

ranking_comparison_df = bm25_results_df.merge(
    final_system_ranking,
    on="Candidate ID",
    how="left"
)

ranking_comparison_df["Rank Improvement"] = (
    ranking_comparison_df["BM25 Rank"]
    -
    ranking_comparison_df["Final Rank"]
)

ranking_comparison_df = (
    ranking_comparison_df
    .sort_values(
        by="Final Rank",
        ascending=True
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 8. Convert ground-truth labels into relevance values
# ---------------------------------------------------------

relevance_map = {
    "High": 3,
    "Medium": 2,
    "Low": 1
}

evaluation_df = (
    cv_data_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Final Score (%)"
        ]
    ]
    .merge(
        bm25_results_df[
            [
                "Candidate ID",
                "BM25 Raw Score"
            ]
        ],
        on="Candidate ID",
        how="left"
    )
)

evaluation_df["Ground Truth Relevance"] = (
    evaluation_df["Ground Truth Label"]
    .map(relevance_map)
)


# ---------------------------------------------------------
# 9. Calculate NDCG ranking performance
# ---------------------------------------------------------

true_relevance = (
    evaluation_df["Ground Truth Relevance"]
    .to_numpy()
    .reshape(1, -1)
)

bm25_prediction_scores = (
    evaluation_df["BM25 Raw Score"]
    .to_numpy()
    .reshape(1, -1)
)

final_prediction_scores = (
    evaluation_df["Final Score (%)"]
    .to_numpy()
    .reshape(1, -1)
)

bm25_ndcg_5 = ndcg_score(
    true_relevance,
    bm25_prediction_scores,
    k=5
)

final_ndcg_5 = ndcg_score(
    true_relevance,
    final_prediction_scores,
    k=5
)

bm25_ndcg_10 = ndcg_score(
    true_relevance,
    bm25_prediction_scores,
    k=10
)

final_ndcg_10 = ndcg_score(
    true_relevance,
    final_prediction_scores,
    k=10
)


# ---------------------------------------------------------
# 10. Calculate Precision@K and Recall@K
# ---------------------------------------------------------

def calculate_high_candidate_metrics(
    results_df,
    score_column,
    k
):
    """
    Treat High-suitability candidates as relevant.
    """

    ranked = (
        results_df
        .sort_values(
            by=score_column,
            ascending=False
        )
        .head(k)
    )

    relevant_in_top_k = (
        ranked["Ground Truth Label"]
        == "High"
    ).sum()

    total_high_candidates = (
        results_df["Ground Truth Label"]
        == "High"
    ).sum()

    precision_at_k = relevant_in_top_k / k

    recall_at_k = (
        relevant_in_top_k
        / total_high_candidates
        if total_high_candidates > 0
        else 0
    )

    return precision_at_k, recall_at_k


bm25_precision_5, bm25_recall_5 = (
    calculate_high_candidate_metrics(
        evaluation_df,
        "BM25 Raw Score",
        5
    )
)

final_precision_5, final_recall_5 = (
    calculate_high_candidate_metrics(
        evaluation_df,
        "Final Score (%)",
        5
    )
)

bm25_precision_10, bm25_recall_10 = (
    calculate_high_candidate_metrics(
        evaluation_df,
        "BM25 Raw Score",
        10
    )
)

final_precision_10, final_recall_10 = (
    calculate_high_candidate_metrics(
        evaluation_df,
        "Final Score (%)",
        10
    )
)


# ---------------------------------------------------------
# 11. Create the performance comparison table
# ---------------------------------------------------------

performance_comparison_df = pd.DataFrame({
    "Metric": [
        "NDCG@5",
        "NDCG@10",
        "Precision@5",
        "Recall@5",
        "Precision@10",
        "Recall@10"
    ],

    "BM25 Baseline": [
        bm25_ndcg_5,
        bm25_ndcg_10,
        bm25_precision_5,
        bm25_recall_5,
        bm25_precision_10,
        bm25_recall_10
    ],

    "Sentence-BERT + Skills": [
        final_ndcg_5,
        final_ndcg_10,
        final_precision_5,
        final_recall_5,
        final_precision_10,
        final_recall_10
    ]
})

performance_comparison_df[
    "Improvement"
] = (
    performance_comparison_df[
        "Sentence-BERT + Skills"
    ]
    -
    performance_comparison_df[
        "BM25 Baseline"
    ]
)

numeric_columns = [
    "BM25 Baseline",
    "Sentence-BERT + Skills",
    "Improvement"
]

performance_comparison_df[numeric_columns] = (
    performance_comparison_df[numeric_columns]
    .round(4)
)


# ---------------------------------------------------------
# 12. Save all Step 7 results
# ---------------------------------------------------------

bm25_output_file = (
    OUTPUT_FOLDER /
    "bm25_baseline_results.csv"
)

comparison_output_file = (
    OUTPUT_FOLDER /
    "bm25_vs_sbert_ranking_comparison.csv"
)

performance_output_file = (
    OUTPUT_FOLDER /
    "ranking_performance_metrics.csv"
)

bm25_results_df.to_csv(
    bm25_output_file,
    index=False
)

ranking_comparison_df.to_csv(
    comparison_output_file,
    index=False
)

performance_comparison_df.to_csv(
    performance_output_file,
    index=False
)


# ---------------------------------------------------------
# 13. Display results
# ---------------------------------------------------------

print("\nBM25 BASELINE RANKING")
display(bm25_results_df)

print("\nBM25 VERSUS FINAL SYSTEM")
display(
    ranking_comparison_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "BM25 Rank",
            "Final Rank",
            "Rank Improvement",
            "BM25 Score (%)",
            "Final Score (%)"
        ]
    ]
)

print("\nPERFORMANCE COMPARISON")
display(performance_comparison_df)


# ---------------------------------------------------------
# 14. Show a simple conclusion
# ---------------------------------------------------------

if final_ndcg_10 > bm25_ndcg_10:
    conclusion = (
        "The Sentence-BERT and structured skill-matching "
        "system achieved better NDCG@10 performance than BM25."
    )

elif final_ndcg_10 < bm25_ndcg_10:
    conclusion = (
        "BM25 achieved better NDCG@10 performance in this "
        "synthetic evaluation. The weights or dataset may "
        "require further review."
    )

else:
    conclusion = (
        "Both systems achieved the same NDCG@10 performance."
    )

print("\nCONCLUSION")
print(conclusion)


# ---------------------------------------------------------
# 15. Final validation
# ---------------------------------------------------------

if (
    len(bm25_results_df) == 20
    and len(ranking_comparison_df) == 20
    and len(performance_comparison_df) == 6
):
    print("\nBM25 comparison completed successfully.")
    print("STEP 7 COMPLETED")

else:
    print(
        "\nWarning: Step 7 did not produce all expected results."
    )

All required data are available.
CVs prepared for BM25: 20
Job-description tokens: 113
BM25 scoring completed.

BM25 BASELINE RANKING


,BM25 Rank,Candidate ID,Ground Truth Label,BM25 Raw Score,BM25 Score (%)
0,1,Candidate_04,High,13.0342,100.00
1,2,Candidate_06,High,11.8441,87.34
2,3,Candidate_03,High,11.8380,87.28
3,4,Candidate_07,High,11.4256,82.89
4,5,Candidate_01,High,11.2061,80.55
5,6,Candidate_05,High,10.2081,69.94
6,7,Candidate_02,High,9.5778,63.23
7,8,Candidate_13,Medium,9.5029,62.43
8,9,Candidate_09,Medium,8.3836,50.53
9,10,Candidate_12,Medium,8.2890,49.52



BM25 VERSUS FINAL SYSTEM


,Candidate ID,Ground Truth Label,BM25 Rank,Final Rank,Rank Improvement,BM25 Score (%),Final Score (%)
0,Candidate_04,High,1,1,0,100.00,80.38
1,Candidate_07,High,4,2,2,82.89,77.10
2,Candidate_06,High,2,3,-1,87.34,75.17
3,Candidate_02,High,7,4,3,63.23,74.56
4,Candidate_01,High,5,5,0,80.55,74.46
5,Candidate_03,High,3,6,-3,87.28,74.08
6,Candidate_05,High,6,7,-1,69.94,72.79
7,Candidate_13,Medium,8,8,0,62.43,72.17
8,Candidate_08,Medium,14,9,5,41.40,71.75
9,Candidate_14,Medium,11,10,1,45.67,71.62



PERFORMANCE COMPARISON


,Metric,BM25 Baseline,Sentence-BERT + Skills,Improvement
0,NDCG@5,1.0000,1.0000,0.0
1,NDCG@10,1.0000,1.0000,0.0
2,Precision@5,1.0000,1.0000,0.0
3,Recall@5,0.7143,0.7143,0.0
4,Precision@10,0.7000,0.7000,0.0
5,Recall@10,1.0000,1.0000,0.0



CONCLUSION
Both systems achieved the same NDCG@10 performance.

BM25 comparison completed successfully.
STEP 7 COMPLETED


In [14]:
# STEP 8: Protected-attribute removal and fairness stability testing

import re
import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Confirm that the required objects are available
# ---------------------------------------------------------

required_objects = [
    "cv_data_df",
    "sbert_model",
    "job_description",
    "required_skills",
    "preprocess_text",
    "calculate_skill_match"
]

missing_objects = [
    item
    for item in required_objects
    if item not in globals()
]

if missing_objects:
    raise ValueError(
        "Missing required objects: "
        + ", ".join(missing_objects)
        + ". Please rerun the earlier steps."
    )

print("All required objects are available.")


# ---------------------------------------------------------
# 2. Use the same weights as the final ranking system
# ---------------------------------------------------------

SEMANTIC_WEIGHT = 0.70
SKILL_WEIGHT = 0.30


# ---------------------------------------------------------
# 3. Prepare the job-description embedding if necessary
# ---------------------------------------------------------

clean_job_description = preprocess_text(
    job_description
)

job_embedding_fairness = sbert_model.encode(
    clean_job_description,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ---------------------------------------------------------
# 4. Define protected-information line patterns
# ---------------------------------------------------------

PROTECTED_ATTRIBUTE_PATTERNS = [
    r"^\s*name\s*:",
    r"^\s*full name\s*:",
    r"^\s*gender\s*:",
    r"^\s*sex\s*:",
    r"^\s*age\s*:",
    r"^\s*date of birth\s*:",
    r"^\s*dob\s*:",
    r"^\s*nationality\s*:",
    r"^\s*ethnicity\s*:",
    r"^\s*religion\s*:",
    r"^\s*marital status\s*:"
]


# ---------------------------------------------------------
# 5. Remove protected information
# ---------------------------------------------------------

def remove_protected_attributes(text):
    """
    Remove clearly labelled protected personal information
    from CV text before candidate scoring.
    """

    if not isinstance(text, str):
        return ""

    remaining_lines = []

    for line in text.splitlines():

        protected_line = any(
            re.match(
                pattern,
                line,
                flags=re.IGNORECASE
            )
            for pattern in PROTECTED_ATTRIBUTE_PATTERNS
        )

        if not protected_line:
            remaining_lines.append(line)

    return "\n".join(remaining_lines).strip()


# ---------------------------------------------------------
# 6. Define two contrasting synthetic profiles
# ---------------------------------------------------------

protected_profile_a = """
Name: Candidate Alpha
Gender: Male
Age: 24
Nationality: Pakistani
Religion: Islam
Marital Status: Single
"""

protected_profile_b = """
Name: Candidate Beta
Gender: Female
Age: 48
Nationality: British
Religion: Christianity
Marital Status: Married
"""


# ---------------------------------------------------------
# 7. Score one CV
# ---------------------------------------------------------

def score_text_for_fairness(
    cv_text,
    remove_protected=True
):
    """
    Calculate semantic, skill and final scores.

    Protected personal information is removed when
    remove_protected=True.
    """

    if remove_protected:
        scoring_text = remove_protected_attributes(
            cv_text
        )
    else:
        scoring_text = cv_text

    clean_text = preprocess_text(
        scoring_text
    )

    candidate_embedding = sbert_model.encode(
        clean_text,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_similarity = float(
        np.dot(
            candidate_embedding,
            job_embedding_fairness
        )
    )

    semantic_score = float(
        np.clip(
            semantic_similarity,
            0,
            1
        ) * 100
    )

    skill_result = calculate_skill_match(
        cv_text=clean_text,
        required_skills=required_skills
    )

    skill_score = float(
        skill_result["Skill Score (%)"]
    )

    final_score = (
        SEMANTIC_WEIGHT * semantic_score
        +
        SKILL_WEIGHT * skill_score
    )

    return {
        "Semantic Score (%)": semantic_score,
        "Skill Score (%)": skill_score,
        "Final Score (%)": final_score
    }


# ---------------------------------------------------------
# 8. Run the fairness stability test on all 20 CVs
# ---------------------------------------------------------

fairness_records = []

for _, row in cv_data_df.iterrows():

    candidate_id = row["Candidate ID"]

    if (
        "Raw Text" in cv_data_df.columns
        and isinstance(row["Raw Text"], str)
        and row["Raw Text"].strip()
    ):
        original_cv_text = row["Raw Text"]

    else:
        original_cv_text = row["Clean Text"]

    # Original candidate score
    original_result = score_text_for_fairness(
        original_cv_text,
        remove_protected=True
    )

    # Add two contrasting protected profiles
    version_a_text = (
        protected_profile_a
        + "\n"
        + original_cv_text
    )

    version_b_text = (
        protected_profile_b
        + "\n"
        + original_cv_text
    )

    # Scores before protected-attribute removal
    version_a_before = score_text_for_fairness(
        version_a_text,
        remove_protected=False
    )

    version_b_before = score_text_for_fairness(
        version_b_text,
        remove_protected=False
    )

    # Scores after protected-attribute removal
    version_a_after = score_text_for_fairness(
        version_a_text,
        remove_protected=True
    )

    version_b_after = score_text_for_fairness(
        version_b_text,
        remove_protected=True
    )

    difference_before = abs(
        version_a_before["Final Score (%)"]
        -
        version_b_before["Final Score (%)"]
    )

    difference_after = abs(
        version_a_after["Final Score (%)"]
        -
        version_b_after["Final Score (%)"]
    )

    original_difference_a = abs(
        original_result["Final Score (%)"]
        -
        version_a_after["Final Score (%)"]
    )

    original_difference_b = abs(
        original_result["Final Score (%)"]
        -
        version_b_after["Final Score (%)"]
    )

    ranking_stable = (
        difference_after <= 0.01
        and original_difference_a <= 0.01
        and original_difference_b <= 0.01
    )

    fairness_records.append({
        "Candidate ID": candidate_id,
        "Ground Truth Label":
            row["Ground Truth Label"],

        "Original Final Score (%)":
            original_result["Final Score (%)"],

        "Profile A Score Before Removal (%)":
            version_a_before["Final Score (%)"],

        "Profile B Score Before Removal (%)":
            version_b_before["Final Score (%)"],

        "Difference Before Removal":
            difference_before,

        "Profile A Score After Removal (%)":
            version_a_after["Final Score (%)"],

        "Profile B Score After Removal (%)":
            version_b_after["Final Score (%)"],

        "Difference After Removal":
            difference_after,

        "Ranking Stable":
            ranking_stable
    })

    print(
        f"{candidate_id}: "
        f"after-removal difference = "
        f"{difference_after:.6f}"
    )


# ---------------------------------------------------------
# 9. Create the fairness-results dataframe
# ---------------------------------------------------------

fairness_results_df = pd.DataFrame(
    fairness_records
)


# ---------------------------------------------------------
# 10. Round values for display
# ---------------------------------------------------------

score_columns = [
    "Original Final Score (%)",
    "Profile A Score Before Removal (%)",
    "Profile B Score Before Removal (%)",
    "Difference Before Removal",
    "Profile A Score After Removal (%)",
    "Profile B Score After Removal (%)",
    "Difference After Removal"
]

fairness_results_df[score_columns] = (
    fairness_results_df[score_columns]
    .round(6)
)


# ---------------------------------------------------------
# 11. Calculate summary results
# ---------------------------------------------------------

stable_candidates = (
    fairness_results_df["Ranking Stable"]
    .sum()
)

total_candidates = len(
    fairness_results_df
)

stability_rate = (
    stable_candidates
    / total_candidates
) * 100

maximum_difference_before = (
    fairness_results_df[
        "Difference Before Removal"
    ].max()
)

maximum_difference_after = (
    fairness_results_df[
        "Difference After Removal"
    ].max()
)

mean_difference_before = (
    fairness_results_df[
        "Difference Before Removal"
    ].mean()
)

mean_difference_after = (
    fairness_results_df[
        "Difference After Removal"
    ].mean()
)


fairness_summary_df = pd.DataFrame({
    "Fairness Measure": [
        "Candidates tested",
        "Stable candidates after removal",
        "Stability rate (%)",
        "Maximum difference before removal",
        "Maximum difference after removal",
        "Mean difference before removal",
        "Mean difference after removal"
    ],

    "Result": [
        total_candidates,
        stable_candidates,
        stability_rate,
        maximum_difference_before,
        maximum_difference_after,
        mean_difference_before,
        mean_difference_after
    ]
})

fairness_summary_df["Result"] = (
    fairness_summary_df["Result"]
    .round(6)
)


# ---------------------------------------------------------
# 12. Save the fairness results
# ---------------------------------------------------------

fairness_output_file = (
    OUTPUT_FOLDER /
    "protected_attribute_fairness_test.csv"
)

fairness_summary_file = (
    OUTPUT_FOLDER /
    "fairness_test_summary.csv"
)

fairness_results_df.to_csv(
    fairness_output_file,
    index=False
)

fairness_summary_df.to_csv(
    fairness_summary_file,
    index=False
)


# ---------------------------------------------------------
# 13. Display results
# ---------------------------------------------------------

print("\nFAIRNESS STABILITY RESULTS")

display(
    fairness_results_df[
        [
            "Candidate ID",
            "Ground Truth Label",
            "Difference Before Removal",
            "Difference After Removal",
            "Ranking Stable"
        ]
    ]
)

print("\nFAIRNESS SUMMARY")

display(fairness_summary_df)

print(
    "\nCandidates with stable scores:",
    f"{stable_candidates}/{total_candidates}"
)

print(
    "Stability rate:",
    f"{stability_rate:.2f}%"
)


# ---------------------------------------------------------
# 14. Interpretation
# ---------------------------------------------------------

if stability_rate == 100:
    print(
        "\nAll candidate scores remained stable after "
        "protected personal information was removed."
    )

    print(
        "The prototype successfully prevented the tested "
        "protected attributes from influencing candidate scores."
    )

else:
    print(
        "\nSome candidate scores changed after protected "
        "information removal."
    )

    print(
        "Review the protected-attribute removal rules before "
        "using the prototype."
    )


# ---------------------------------------------------------
# 15. Final validation
# ---------------------------------------------------------

if (
    len(fairness_results_df) == 20
    and fairness_results_df[
        "Ranking Stable"
    ].all()
):
    print("\nFairness stability testing completed successfully.")
    print("STEP 8 COMPLETED")

else:
    print(
        "\nStep 8 completed, but some fairness-stability "
        "checks require review."
    )

All required objects are available.
Candidate_01: after-removal difference = 0.000000
Candidate_02: after-removal difference = 0.000000
Candidate_03: after-removal difference = 0.000000
Candidate_04: after-removal difference = 0.000000
Candidate_05: after-removal difference = 0.000000
Candidate_06: after-removal difference = 0.000000
Candidate_07: after-removal difference = 0.000000
Candidate_08: after-removal difference = 0.000000
Candidate_09: after-removal difference = 0.000000
Candidate_10: after-removal difference = 0.000000
Candidate_11: after-removal difference = 0.000000
Candidate_12: after-removal difference = 0.000000
Candidate_13: after-removal difference = 0.000000
Candidate_14: after-removal difference = 0.000000
Candidate_15: after-removal difference = 0.000000
Candidate_16: after-removal difference = 0.000000
Candidate_17: after-removal difference = 0.000000
Candidate_18: after-removal difference = 0.000000
Candidate_19: after-removal difference = 0.000000
Candidate_20: 

,Candidate ID,Ground Truth Label,Difference Before Removal,Difference After Removal,Ranking Stable
0,Candidate_01,High,1.546226,0.0,True
1,Candidate_02,High,1.658061,0.0,True
2,Candidate_03,High,1.923805,0.0,True
3,Candidate_04,High,1.985968,0.0,True
4,Candidate_05,High,1.636578,0.0,True
5,Candidate_06,High,1.801710,0.0,True
6,Candidate_07,High,1.975200,0.0,True
7,Candidate_08,Medium,1.783252,0.0,True
8,Candidate_09,Medium,2.414525,0.0,True
9,Candidate_10,Medium,2.251153,0.0,True



FAIRNESS SUMMARY


,Fairness Measure,Result
0,Candidates tested,20.000000
1,Stable candidates after removal,20.000000
2,Stability rate (%),100.000000
3,Maximum difference before removal,3.011628
4,Maximum difference after removal,0.000000
5,Mean difference before removal,1.881455
6,Mean difference after removal,0.000000



Candidates with stable scores: 20/20
Stability rate: 100.00%

All candidate scores remained stable after protected personal information was removed.
The prototype successfully prevented the tested protected attributes from influencing candidate scores.

Fairness stability testing completed successfully.
STEP 8 COMPLETED


In [15]:
%%writefile /content/cv_job_matcher/app.py

import re
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

from docx import Document
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


# =========================================================
# PAGE SETTINGS
# =========================================================

st.set_page_config(
    page_title="Explainable AI Candidate Matcher",
    page_icon="📄",
    layout="wide"
)


# =========================================================
# PROJECT PATHS
# =========================================================

BASE_FOLDER = Path("/content/cv_job_matcher")
OUTPUT_FOLDER = BASE_FOLDER / "outputs"

JOB_DESCRIPTION_FILE = (
    BASE_FOLDER /
    "software_engineer_job_description.txt"
)

FINAL_RESULTS_FILE = (
    OUTPUT_FOLDER /
    "final_explainable_candidate_ranking.csv"
)


# =========================================================
# SYSTEM SETTINGS
# =========================================================

MODEL_NAME = "paraphrase-MiniLM-L3-v2"

SEMANTIC_WEIGHT = 0.70
SKILL_WEIGHT = 0.30


# =========================================================
# REQUIRED SKILLS
# =========================================================

REQUIRED_SKILLS = [
    "Python",
    "SQL",
    "Object-Oriented Programming",
    "REST APIs",
    "Git",
    "Docker",
    "AWS",
    "Machine Learning",
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "Software Testing",
    "Agile"
]


SKILL_ALIASES = {
    "Python": [
        "python"
    ],

    "SQL": [
        "sql",
        "mysql",
        "postgresql",
        "structured query language"
    ],

    "Object-Oriented Programming": [
        "object-oriented programming",
        "object oriented programming",
        "oop"
    ],

    "REST APIs": [
        "rest api",
        "rest apis",
        "restful api",
        "restful apis",
        "api development"
    ],

    "Git": [
        "git",
        "github",
        "gitlab",
        "version control"
    ],

    "Docker": [
        "docker",
        "containerisation",
        "containerization",
        "containerised",
        "containerized"
    ],

    "AWS": [
        "aws",
        "amazon web services"
    ],

    "Machine Learning": [
        "machine learning",
        "predictive modelling",
        "predictive modeling"
    ],

    "Pandas": [
        "pandas"
    ],

    "NumPy": [
        "numpy"
    ],

    "Scikit-learn": [
        "scikit-learn",
        "scikit learn",
        "sklearn"
    ],

    "Software Testing": [
        "software testing",
        "unit testing",
        "integration testing",
        "test automation",
        "testing and debugging"
    ],

    "Agile": [
        "agile",
        "scrum",
        "sprint planning"
    ]
}


# =========================================================
# LOAD MODEL
# =========================================================

@st.cache_resource
def load_sbert_model():
    return SentenceTransformer(
        MODEL_NAME,
        device="cpu"
    )


# =========================================================
# LOAD EXISTING RESULTS
# =========================================================

@st.cache_data
def load_existing_results():
    if FINAL_RESULTS_FILE.exists():
        return pd.read_csv(FINAL_RESULTS_FILE)

    return pd.DataFrame()


# =========================================================
# LOAD DEFAULT JOB DESCRIPTION
# =========================================================

@st.cache_data
def load_default_job_description():
    if JOB_DESCRIPTION_FILE.exists():
        return JOB_DESCRIPTION_FILE.read_text(
            encoding="utf-8"
        )

    return ""


# =========================================================
# TEXT EXTRACTION
# =========================================================

def extract_pdf_text(uploaded_file):
    uploaded_file.seek(0)

    reader = PdfReader(uploaded_file)
    pages = []

    for page in reader.pages:
        page_text = page.extract_text() or ""

        if page_text.strip():
            pages.append(page_text.strip())

    return "\n".join(pages)


def extract_docx_text(uploaded_file):
    uploaded_file.seek(0)

    document = Document(uploaded_file)

    paragraphs = [
        paragraph.text.strip()
        for paragraph in document.paragraphs
        if paragraph.text.strip()
    ]

    return "\n".join(paragraphs)


def extract_uploaded_cv_text(uploaded_file):
    filename = uploaded_file.name.lower()

    if filename.endswith(".pdf"):
        text = extract_pdf_text(uploaded_file)

    elif filename.endswith(".docx"):
        text = extract_docx_text(uploaded_file)

    else:
        raise ValueError(
            "Only PDF and DOCX files are supported."
        )

    if len(text.strip()) < 50:
        raise ValueError(
            "The CV contains insufficient readable text. "
            "It may be a scanned image-based PDF."
        )

    return text.strip()


# =========================================================
# PROTECTED-ATTRIBUTE REMOVAL
# =========================================================

PROTECTED_ATTRIBUTE_PATTERNS = [
    r"^\s*name\s*:",
    r"^\s*full name\s*:",
    r"^\s*gender\s*:",
    r"^\s*sex\s*:",
    r"^\s*age\s*:",
    r"^\s*date of birth\s*:",
    r"^\s*dob\s*:",
    r"^\s*nationality\s*:",
    r"^\s*ethnicity\s*:",
    r"^\s*religion\s*:",
    r"^\s*marital status\s*:"
]


def remove_protected_attributes(text):
    remaining_lines = []

    for line in text.splitlines():

        is_protected = any(
            re.match(
                pattern,
                line,
                flags=re.IGNORECASE
            )
            for pattern in PROTECTED_ATTRIBUTE_PATTERNS
        )

        if not is_protected:
            remaining_lines.append(line)

    return "\n".join(remaining_lines).strip()


# =========================================================
# TEXT PREPROCESSING
# =========================================================

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"\+?\d[\d\s().-]{7,}\d",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z0-9+#.\-/\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# =========================================================
# SKILL MATCHING
# =========================================================

def skill_is_present(cv_text, skill):
    aliases = SKILL_ALIASES.get(
        skill,
        [skill.lower()]
    )

    for alias in aliases:

        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(alias.lower())
            + r"(?![a-z0-9])"
        )

        if re.search(pattern, cv_text):
            return True

    return False


def calculate_skill_match(
    cv_text,
    required_skills
):
    matched_skills = []
    missing_skills = []

    for skill in required_skills:

        if skill_is_present(cv_text, skill):
            matched_skills.append(skill)

        else:
            missing_skills.append(skill)

    if required_skills:
        skill_score = (
            len(matched_skills)
            / len(required_skills)
        ) * 100

    else:
        skill_score = 0.0

    return {
        "skill_score": skill_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills
    }


# =========================================================
# SUITABILITY CLASSIFICATION
# =========================================================

def classify_suitability(final_score):
    if final_score >= 70:
        return "High"

    if final_score >= 50:
        return "Medium"

    return "Low"


def create_recommendation(label):
    if label == "High":
        return (
            "Strong candidate for initial recruiter review."
        )

    if label == "Medium":
        return (
            "Potential candidate, but missing requirements "
            "should be reviewed."
        )

    return (
        "Limited match for this role based on the current "
        "job description."
    )


# =========================================================
# SCORE A NEW CV
# =========================================================

def score_new_cv(
    cv_text,
    job_description,
    required_skills
):
    protected_removed_text = (
        remove_protected_attributes(cv_text)
    )

    clean_cv_text = preprocess_text(
        protected_removed_text
    )

    clean_job_description = preprocess_text(
        job_description
    )

    if not clean_cv_text:
        raise ValueError(
            "The CV contains no readable text after processing."
        )

    if not clean_job_description:
        raise ValueError(
            "The job description is empty."
        )

    model = load_sbert_model()

    embeddings = model.encode(
        [
            clean_job_description,
            clean_cv_text
        ],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_similarity = float(
        np.dot(
            embeddings[0],
            embeddings[1]
        )
    )

    semantic_score = float(
        np.clip(
            semantic_similarity,
            0,
            1
        ) * 100
    )

    skill_result = calculate_skill_match(
        clean_cv_text,
        required_skills
    )

    skill_score = skill_result["skill_score"]

    final_score = (
        SEMANTIC_WEIGHT * semantic_score
        +
        SKILL_WEIGHT * skill_score
    )

    final_score = float(
        np.clip(
            final_score,
            0,
            100
        )
    )

    predicted_suitability = (
        classify_suitability(final_score)
    )

    recommendation = create_recommendation(
        predicted_suitability
    )

    explanation = (
        f"The CV received a semantic similarity score of "
        f"{semantic_score:.2f}% and a structured skill-match "
        f"score of {skill_score:.2f}%. The final score uses "
        f"70% semantic similarity and 30% structured skill "
        f"matching. The predicted suitability is "
        f"{predicted_suitability}."
    )

    return {
        "semantic_score": semantic_score,
        "skill_score": skill_score,
        "final_score": final_score,
        "predicted_suitability":
            predicted_suitability,
        "matched_skills":
            skill_result["matched_skills"],
        "missing_skills":
            skill_result["missing_skills"],
        "recommendation": recommendation,
        "explanation": explanation,
        "processed_text": clean_cv_text
    }


# =========================================================
# WEBSITE HEADER
# =========================================================

st.title(
    "Explainable AI Candidate–Job Matching Prototype"
)

st.write(
    "This prototype combines Sentence-BERT semantic "
    "similarity with structured skill matching to support "
    "initial CV screening."
)

st.warning(
    "This system provides decision support only. "
    "Final recruitment decisions must be made by humans."
)


# =========================================================
# WEBSITE TABS
# =========================================================

tab1, tab2, tab3 = st.tabs([
    "Existing 20-CV Ranking",
    "Check a New CV",
    "System Information"
])


# =========================================================
# TAB 1: EXISTING RANKING
# =========================================================

with tab1:

    st.header("Existing Synthetic Candidate Ranking")

    existing_results_df = load_existing_results()

    if existing_results_df.empty:

        st.error(
            "The final ranking file was not found. "
            "Please make sure Step 6 was completed."
        )

    else:

        total_candidates = len(
            existing_results_df
        )

        high_candidates = (
            existing_results_df[
                "Predicted Suitability"
            ]
            .eq("High")
            .sum()
        )

        medium_candidates = (
            existing_results_df[
                "Predicted Suitability"
            ]
            .eq("Medium")
            .sum()
        )

        low_candidates = (
            existing_results_df[
                "Predicted Suitability"
            ]
            .eq("Low")
            .sum()
        )

        col1, col2, col3, col4 = st.columns(4)

        col1.metric(
            "Total Candidates",
            total_candidates
        )

        col2.metric(
            "High Suitability",
            high_candidates
        )

        col3.metric(
            "Medium Suitability",
            medium_candidates
        )

        col4.metric(
            "Low Suitability",
            low_candidates
        )

        display_columns = [
            column
            for column in [
                "Final Rank",
                "Candidate ID",
                "Ground Truth Label",
                "Semantic Score (%)",
                "Skill Score (%)",
                "Final Score (%)",
                "Predicted Suitability",
                "Recommendation"
            ]
            if column in existing_results_df.columns
        ]

        st.dataframe(
            existing_results_df[
                display_columns
            ],
            use_container_width=True,
            hide_index=True
        )

        candidate_options = (
            existing_results_df[
                "Candidate ID"
            ]
            .astype(str)
            .tolist()
        )

        selected_candidate = st.selectbox(
            "Select a candidate to view the explanation",
            candidate_options
        )

        selected_row = existing_results_df[
            existing_results_df["Candidate ID"]
            == selected_candidate
        ].iloc[0]

        st.subheader(
            f"Explanation for {selected_candidate}"
        )

        if "Explanation" in selected_row.index:
            st.info(
                selected_row["Explanation"]
            )

        if "Matched Skills" in selected_row.index:
            st.write(
                "**Matched skills:**",
                selected_row["Matched Skills"]
            )

        if "Missing Skills" in selected_row.index:
            st.write(
                "**Missing skills:**",
                selected_row["Missing Skills"]
            )


# =========================================================
# TAB 2: CHECK A NEW CV
# =========================================================

with tab2:

    st.header("Check a New Candidate CV")

    default_job_description = (
        load_default_job_description()
    )

    job_description_input = st.text_area(
        "Job description",
        value=default_job_description,
        height=280
    )

    skills_input = st.text_area(
        "Required skills — one skill per line",
        value="\n".join(REQUIRED_SKILLS),
        height=220
    )

    new_required_skills = [
        skill.strip()
        for skill in skills_input.splitlines()
        if skill.strip()
    ]

    uploaded_cv = st.file_uploader(
        "Upload a CV",
        type=[
            "pdf",
            "docx"
        ],
        accept_multiple_files=False
    )

    analyse_button = st.button(
        "Analyse New CV",
        type="primary"
    )

    if analyse_button:

        if uploaded_cv is None:

            st.error(
                "Please upload a PDF or DOCX CV."
            )

        elif not job_description_input.strip():

            st.error(
                "Please provide a job description."
            )

        elif not new_required_skills:

            st.error(
                "Please provide at least one required skill."
            )

        else:

            try:

                with st.spinner(
                    "Extracting and analysing the CV..."
                ):

                    extracted_cv_text = (
                        extract_uploaded_cv_text(
                            uploaded_cv
                        )
                    )

                    new_result = score_new_cv(
                        cv_text=extracted_cv_text,
                        job_description=
                            job_description_input,
                        required_skills=
                            new_required_skills
                    )

                st.success(
                    "CV analysis completed successfully."
                )

                col1, col2, col3 = st.columns(3)

                col1.metric(
                    "Semantic Similarity",
                    f"{new_result['semantic_score']:.2f}%"
                )

                col2.metric(
                    "Skill Match",
                    f"{new_result['skill_score']:.2f}%"
                )

                col3.metric(
                    "Final Score",
                    f"{new_result['final_score']:.2f}%"
                )

                st.subheader(
                    "Predicted Suitability"
                )

                if (
                    new_result[
                        "predicted_suitability"
                    ]
                    == "High"
                ):
                    st.success("High suitability")

                elif (
                    new_result[
                        "predicted_suitability"
                    ]
                    == "Medium"
                ):
                    st.warning("Medium suitability")

                else:
                    st.error("Low suitability")

                st.info(
                    new_result["recommendation"]
                )

                left_column, right_column = (
                    st.columns(2)
                )

                with left_column:

                    st.subheader("Matched Skills")

                    if new_result["matched_skills"]:

                        for skill in (
                            new_result[
                                "matched_skills"
                            ]
                        ):
                            st.write(f"✅ {skill}")

                    else:
                        st.write(
                            "No required skills were identified."
                        )

                with right_column:

                    st.subheader("Missing Skills")

                    if new_result["missing_skills"]:

                        for skill in (
                            new_result[
                                "missing_skills"
                            ]
                        ):
                            st.write(f"❌ {skill}")

                    else:
                        st.write(
                            "No required skills are missing."
                        )

                st.subheader(
                    "Explainable Recommendation"
                )

                st.write(
                    new_result["explanation"]
                )

                existing_results_df = (
                    load_existing_results()
                )

                if not existing_results_df.empty:

                    new_candidate_row = pd.DataFrame([
                        {
                            "Candidate ID":
                                uploaded_cv.name,

                            "Ground Truth Label":
                                "New CV",

                            "Semantic Score (%)":
                                round(
                                    new_result[
                                        "semantic_score"
                                    ],
                                    2
                                ),

                            "Skill Score (%)":
                                round(
                                    new_result[
                                        "skill_score"
                                    ],
                                    2
                                ),

                            "Final Score (%)":
                                round(
                                    new_result[
                                        "final_score"
                                    ],
                                    2
                                ),

                            "Predicted Suitability":
                                new_result[
                                    "predicted_suitability"
                                ],

                            "Recommendation":
                                new_result[
                                    "recommendation"
                                ]
                        }
                    ])

                    ranking_columns = [
                        "Candidate ID",
                        "Ground Truth Label",
                        "Semantic Score (%)",
                        "Skill Score (%)",
                        "Final Score (%)",
                        "Predicted Suitability",
                        "Recommendation"
                    ]

                    existing_for_ranking = (
                        existing_results_df[
                            ranking_columns
                        ].copy()
                    )

                    updated_ranking = pd.concat(
                        [
                            existing_for_ranking,
                            new_candidate_row
                        ],
                        ignore_index=True
                    )

                    updated_ranking = (
                        updated_ranking
                        .sort_values(
                            by="Final Score (%)",
                            ascending=False
                        )
                        .reset_index(drop=True)
                    )

                    updated_ranking.insert(
                        0,
                        "Rank",
                        range(
                            1,
                            len(updated_ranking) + 1
                        )
                    )

                    new_cv_rank = int(
                        updated_ranking.loc[
                            updated_ranking[
                                "Candidate ID"
                            ]
                            == uploaded_cv.name,
                            "Rank"
                        ].iloc[0]
                    )

                    st.subheader(
                        "Position Among Existing Candidates"
                    )

                    st.metric(
                        "New CV Rank",
                        f"{new_cv_rank} of "
                        f"{len(updated_ranking)}"
                    )

                    st.dataframe(
                        updated_ranking,
                        use_container_width=True,
                        hide_index=True
                    )

                with st.expander(
                    "View extracted CV text"
                ):
                    st.text(
                        extracted_cv_text[:15000]
                    )

            except Exception as error:

                st.error(
                    f"CV analysis failed: {error}"
                )


# =========================================================
# TAB 3: SYSTEM INFORMATION
# =========================================================

with tab3:

    st.header("System Information")

    st.write(
        "**Semantic model:** "
        f"{MODEL_NAME}"
    )

    st.write(
        "**Semantic similarity weight:** "
        f"{SEMANTIC_WEIGHT * 100:.0f}%"
    )

    st.write(
        "**Structured skill weight:** "
        f"{SKILL_WEIGHT * 100:.0f}%"
    )

    st.write(
        "**Supported CV formats:** PDF and DOCX"
    )

    st.write(
        "**Protected information:** Clearly labelled age, "
        "gender, nationality, religion and similar fields are "
        "removed before scoring."
    )

    st.write(
        "**Purpose:** Initial candidate-screening decision "
        "support with mandatory human oversight."
    )

Writing /content/cv_job_matcher/app.py


In [16]:
# STEP 10.1: Install Cloudflare Tunnel

!wget -q \
https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
-O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared

!cloudflared --version

print("STEP 10.1 COMPLETED")

cloudflared version 2026.7.1 (built 2026-07-09-13:00 UTC)
STEP 10.1 COMPLETED


In [17]:
# STEP 10.2: Start the Streamlit application

import os
import time
import requests

APP_PATH = "/content/cv_job_matcher/app.py"
STREAMLIT_LOG = "/content/streamlit.log"

# Stop any previous Streamlit process
os.system("pkill -f 'streamlit run' >/dev/null 2>&1")

# Start the application in the background
os.system(
    f"nohup streamlit run {APP_PATH} "
    "--server.port 8501 "
    "--server.address 0.0.0.0 "
    "--server.headless true "
    f"> {STREAMLIT_LOG} 2>&1 &"
)

print("Starting Streamlit...")

app_ready = False

for attempt in range(30):

    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )

        if response.status_code == 200:
            app_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)

if app_ready:
    print("Streamlit application is running.")
    print("Local address: http://127.0.0.1:8501")
    print("STEP 10.2 COMPLETED")

else:
    print("Streamlit did not start correctly.")
    print("\nApplication log:\n")

    if os.path.exists(STREAMLIT_LOG):
        with open(
            STREAMLIT_LOG,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as log_file:
            print(log_file.read())

Starting Streamlit...
Streamlit application is running.
Local address: http://127.0.0.1:8501
STEP 10.2 COMPLETED


In [18]:
# STEP 10.3: Create a temporary public website link

import os
import re
import time
from pathlib import Path

CLOUDFLARE_LOG = Path(
    "/content/cloudflared.log"
)

# Stop any previous Cloudflare tunnel
os.system(
    "pkill -f 'cloudflared tunnel' "
    ">/dev/null 2>&1"
)

# Remove the previous log
if CLOUDFLARE_LOG.exists():
    CLOUDFLARE_LOG.unlink()

# Start a new tunnel in the background
os.system(
    "nohup cloudflared tunnel "
    "--url http://127.0.0.1:8501 "
    "--no-autoupdate "
    "> /content/cloudflared.log 2>&1 &"
)

print("Creating public link...")

public_url = None

for attempt in range(40):

    if CLOUDFLARE_LOG.exists():

        log_text = CLOUDFLARE_LOG.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        matches = re.findall(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            log_text
        )

        if matches:
            public_url = matches[0]
            break

    time.sleep(2)

if public_url:

    print("\nPROTOTYPE WEBSITE IS READY")
    print("--------------------------")
    print(public_url)
    print("\nOpen this link in a new browser tab.")
    print("STEP 10 COMPLETED")

else:

    print("The public link was not generated.")
    print("\nCloudflare log:\n")

    if CLOUDFLARE_LOG.exists():
        print(
            CLOUDFLARE_LOG.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

Creating public link...

PROTOTYPE WEBSITE IS READY
--------------------------
https://which-coupled-relative-shopping.trycloudflare.com

Open this link in a new browser tab.
STEP 10 COMPLETED


In [19]:
# RESTART STREAMLIT AND CREATE A NEW WORKING LINK

import os
import re
import time
import requests
from pathlib import Path


APP_PATH = Path("/content/cv_job_matcher/app.py")
STREAMLIT_LOG = Path("/content/streamlit.log")
CLOUDFLARE_LOG = Path("/content/cloudflared.log")


# ---------------------------------------------------------
# 1. Confirm that app.py still exists
# ---------------------------------------------------------

if not APP_PATH.exists():
    raise FileNotFoundError(
        "app.py is missing. Your Colab runtime may have restarted. "
        "Rerun Step 9 to recreate the Streamlit application."
    )

print("Application file found.")


# ---------------------------------------------------------
# 2. Install cloudflared if it is missing
# ---------------------------------------------------------

if not Path("/usr/local/bin/cloudflared").exists():

    print("Installing Cloudflare Tunnel...")

    os.system(
        "wget -q "
        "https://github.com/cloudflare/cloudflared/releases/latest/"
        "download/cloudflared-linux-amd64 "
        "-O /usr/local/bin/cloudflared"
    )

    os.system(
        "chmod +x /usr/local/bin/cloudflared"
    )

print("Cloudflare Tunnel is available.")


# ---------------------------------------------------------
# 3. Stop previous processes
# ---------------------------------------------------------

os.system(
    "pkill -f 'streamlit run' >/dev/null 2>&1"
)

os.system(
    "pkill -f 'cloudflared tunnel' >/dev/null 2>&1"
)

time.sleep(2)


# ---------------------------------------------------------
# 4. Remove old logs
# ---------------------------------------------------------

for log_file in [
    STREAMLIT_LOG,
    CLOUDFLARE_LOG
]:
    if log_file.exists():
        log_file.unlink()


# ---------------------------------------------------------
# 5. Start Streamlit
# ---------------------------------------------------------

print("Starting Streamlit...")

os.system(
    f"nohup python -m streamlit run {APP_PATH} "
    "--server.port 8501 "
    "--server.address 0.0.0.0 "
    "--server.headless true "
    "--browser.gatherUsageStats false "
    f"> {STREAMLIT_LOG} 2>&1 &"
)


# ---------------------------------------------------------
# 6. Wait for Streamlit health check
# ---------------------------------------------------------

streamlit_ready = False

for attempt in range(60):

    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=3
        )

        if response.status_code == 200:
            streamlit_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)


if not streamlit_ready:

    print("\nSTREAMLIT FAILED TO START")

    if STREAMLIT_LOG.exists():
        print(STREAMLIT_LOG.read_text(
            encoding="utf-8",
            errors="ignore"
        ))

    raise RuntimeError(
        "Streamlit is not running."
    )


print("Streamlit is running successfully.")


# ---------------------------------------------------------
# 7. Start a new Cloudflare tunnel
# ---------------------------------------------------------

print("Creating a new public link...")

os.system(
    "nohup /usr/local/bin/cloudflared tunnel "
    "--url http://127.0.0.1:8501 "
    "--protocol http2 "
    "--no-autoupdate "
    f"> {CLOUDFLARE_LOG} 2>&1 &"
)


# ---------------------------------------------------------
# 8. Retrieve the new public link
# ---------------------------------------------------------

public_url = None

for attempt in range(60):

    if CLOUDFLARE_LOG.exists():

        log_text = CLOUDFLARE_LOG.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        matches = re.findall(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            log_text
        )

        if matches:
            public_url = matches[-1]
            break

    time.sleep(2)


if public_url:

    print("\nNEW PROTOTYPE LINK")
    print("==============================")
    print(public_url)
    print("==============================")
    print("\nOpen this NEW link, not the previous link.")

else:

    print("\nCLOUDFLARE FAILED TO CREATE A LINK")

    if CLOUDFLARE_LOG.exists():
        print(CLOUDFLARE_LOG.read_text(
            encoding="utf-8",
            errors="ignore"
        ))

Application file found.
Cloudflare Tunnel is available.
Starting Streamlit...
Streamlit is running successfully.
Creating a new public link...

NEW PROTOTYPE LINK
https://stroke-hay-sellers-mechanics.trycloudflare.com

Open this NEW link, not the previous link.


In [20]:
# STEP 11.1: Create a new CV for testing the website

from docx import Document
from pathlib import Path

test_cv_path = Path("/content/New_Test_Candidate_CV.docx")

document = Document()

document.add_heading("New Test Candidate", level=0)

document.add_heading("Professional Profile", level=1)
document.add_paragraph(
    "Software developer with three years of experience building "
    "Python applications, REST APIs and database-driven systems. "
    "Experienced in software testing, debugging, Git version control "
    "and Agile development."
)

document.add_heading("Technical Skills", level=1)

test_skills = [
    "Python",
    "SQL",
    "Object-Oriented Programming",
    "REST APIs",
    "Git",
    "Docker",
    "Flask",
    "PostgreSQL",
    "Software Testing",
    "Agile"
]

for skill in test_skills:
    document.add_paragraph(
        skill,
        style="List Bullet"
    )

document.add_heading("Professional Experience", level=1)

document.add_paragraph(
    "Junior Software Developer — Example Technology Company"
)

experience_points = [
    "Developed Python applications using object-oriented programming.",
    "Created REST APIs for internal software systems.",
    "Worked with SQL and PostgreSQL databases.",
    "Used Git and GitHub for source-code management.",
    "Performed software testing, debugging and code reviews.",
    "Participated in Agile sprint planning and team meetings.",
    "Used Docker to run development applications."
]

for point in experience_points:
    document.add_paragraph(
        point,
        style="List Bullet"
    )

document.add_heading("Projects", level=1)

document.add_paragraph(
    "Inventory Management System: Developed a Python and SQL "
    "application for recording products, stock levels and transactions.",
    style="List Bullet"
)

document.add_paragraph(
    "REST API Project: Developed an API using Flask and PostgreSQL.",
    style="List Bullet"
)

document.add_heading("Education", level=1)
document.add_paragraph(
    "BSc Computer Science"
)

document.add_heading("Additional Information", level=1)
document.add_paragraph(
    "This is a synthetic CV created only for prototype testing."
)

document.save(test_cv_path)

print("New test CV created successfully.")
print("File location:", test_cv_path)
print("STEP 11.1 COMPLETED")

New test CV created successfully.
File location: /content/New_Test_Candidate_CV.docx
STEP 11.1 COMPLETED


In [21]:
# STEP 11.2: Download the new test CV

from google.colab import files

files.download(
    "/content/New_Test_Candidate_CV.docx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>